In [0]:
MERGE INTO dev.demo.orders AS t
USING dev.demo.raw_orders AS s
ON t.order_id = s.order_id AND t.is_current = true
WHEN MATCHED AND t.status != s.status THEN
  UPDATE SET
    end_date    = current_date(),
    is_current = false;

-- Step 2: Insert new records and new versions of changed records
INSERT INTO dev.demo.orders
SELECT
  s.order_id,
  s.status,
  current_date()           AS start_date,
  NULL                      AS end_date,
  true                      AS is_current,
  COALESCE(v.max_version, 0) + 1 AS version
FROM dev.demo.raw_orders AS s
LEFT JOIN (
  SELECT order_id, MAX(version) AS max_version
  FROM dev.demo.orders
  GROUP BY order_id
) AS v
  ON s.order_id = v.order_id
WHERE NOT EXISTS (
  SELECT 1
  FROM dev.demo.orders AS o
  WHERE o.order_id    = s.order_id
    AND o.is_current  = true
    AND o.status      = s.status
);

